<h1>Exploratory Data analysis on log traces</h1>

In [29]:
import pandas as pd
import json
from sklearn.utils import resample
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import torch.nn as nn
import torch.optim as optim
import os
import json

In [16]:
FAIL_CSV = '../../data/raw/failure_trace.csv'
NORM_CSV = '../../data/raw/normal_trace.csv'
FAIL_JSON = '../../data/raw/failure_taskId.json'
NORM_JSON = '../../data/raw/normal_taskId.json'

In [17]:
with open(FAIL_CSV, 'r') as f:
    fail_lines = f.read().splitlines()
df_fail = pd.DataFrame(fail_lines, columns=['raw_trace'])

with open(NORM_CSV, 'r') as f:
    norm_lines = f.read().splitlines()
df_norm = pd.DataFrame(norm_lines, columns=['raw_trace'])

In [18]:
# 2. Split the TaskID from the sequence for both DataFrames
df_fail[['TaskID', 'Trace_Sequence']] = df_fail['raw_trace'].str.split(',', n=1, expand=True)
df_norm[['TaskID', 'Trace_Sequence']] = df_norm['raw_trace'].str.split(',', n=1, expand=True)

In [19]:
# ==========================================
# STEP 3: Load Valid IDs & Hardcode Binary Labels
# ==========================================
# Extract ONLY the TaskID (item[0]) from the JSON, ignoring the useless index (item[1])
with open(FAIL_JSON, 'r') as f:
    fail_ids = [item[0] for item in json.load(f)]
    
with open(NORM_JSON, 'r') as f:
    norm_ids = [item[0] for item in json.load(f)]

# ==========================================
# STEP 4: Filter and Label
# ==========================================
# Keep only the rows where the TaskID exists in our JSON lists
df_fail_filtered = df_fail[df_fail['TaskID'].isin(fail_ids)].copy()
df_norm_filtered = df_norm[df_norm['TaskID'].isin(norm_ids)].copy()

# Manually assign our Binary Labels!
df_fail_filtered['Label'] = 1  # 1 = Anomaly
df_norm_filtered['Label'] = 0  # 0 = Normal

# ==========================================
# STEP 5 & 6: Combine and Shuffle
# ==========================================
df_master = pd.concat([df_fail_filtered, df_norm_filtered], ignore_index=True)

# Shuffle the dataset so the LSTM doesn't memorize the order
df_master = df_master.sample(frac=1, random_state=42).reset_index(drop=True)

print("--- CORRECTED BINARY DATASET ---")
print(f"Total Traces ready for LSTM: {len(df_master)}")
print(f"Label Distribution:\n{df_master['Label'].value_counts()}")

--- CORRECTED BINARY DATASET ---
Total Traces ready for LSTM: 256584
Label Distribution:
Label
0    226767
1     29817
Name: count, dtype: int64


In [20]:
# ==========================================
# STEP 7: Handle Class Imbalance (Downsampling)
# ==========================================
# 1. Separate the majority and minority classes
df_majority = df_master[df_master['Label'] == 0]
df_minority = df_master[df_master['Label'] == 1]

# 2. Downsample the majority class to match the minority class
df_majority_downsampled = resample(df_majority, 
                                 replace=False,     # sample without replacement
                                 n_samples=len(df_minority), # match the minority count (29,817)
                                 random_state=42)   # reproducible results

# 3. Combine the downsampled majority class with the original minority class
df_balanced = pd.concat([df_majority_downsampled, df_minority])

# 4. Shuffle the new balanced dataset
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("--- STEP 7: BALANCED DATASET ---")
print(f"Total Traces ready for PyTorch: {len(df_balanced)}")
print(f"New Label Distribution:\n{df_balanced['Label'].value_counts()}")

--- STEP 7: BALANCED DATASET ---
Total Traces ready for PyTorch: 59634
New Label Distribution:
Label
0    29817
1    29817
Name: count, dtype: int64


In [21]:
df_balanced.head()

,raw_trace,TaskID,Trace_Sequence,Label
0,"1C306D733C21EFC4,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0...",1C306D733C21EFC4,"1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....",0
1,"6E423AC77A4D8E9F,4.0,4.0,2.0,2.0,17.0,17.0,16....",6E423AC77A4D8E9F,"4.0,4.0,2.0,2.0,17.0,17.0,16.0,16.0,16.0,10.0,...",1
2,"62BE5DB8D5B9244E,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0...",62BE5DB8D5B9244E,"1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....",0
3,"005D315F0148CFCC,4.0,4.0,1.0,1.0,9.0,9.0,6.0,6...",005D315F0148CFCC,"4.0,4.0,1.0,1.0,9.0,9.0,6.0,6.0,6.0,2.0,4.0,4....",1
4,"784FABAED60B14A8,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0...",784FABAED60B14A8,"2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....",0


In [22]:
# ==========================================
# STEP 8 & 9: Clean Floats & Pad Sequences
# ==========================================
MAX_SEQ_LEN = 100 # Force every sequence to be exactly 100 steps long

def clean_and_pad_pretokenized(seq_string):
    # 1. Split the string by commas
    # 2. Convert the string to a float (e.g. "4.0")
    # 3. Convert the float to a strict integer (e.g. 4)
    # We use a try/except to catch any trailing commas or empty spaces
    tokens = []
    for x in str(seq_string).split(','):
        x = x.strip()
        if x:  # If the string is not empty
            try:
                tokens.append(int(float(x)))
            except ValueError:
                pass # Skip any weird text artifacts

    # Pad with 0s if too short, truncate if too long
    if len(tokens) > MAX_SEQ_LEN:
        return tokens[:MAX_SEQ_LEN]
    else:
        return tokens + [0] * (MAX_SEQ_LEN - len(tokens))
# Apply the cleaning function to our balanced DataFrame
df_balanced['Encoded_Sequence'] = df_balanced['Trace_Sequence'].apply(clean_and_pad_pretokenized)

In [23]:
# ==========================================
# STEP 10: Convert to PyTorch Tensors
# ==========================================
# X MUST be torch.long (integers) so the PyTorch Embedding layer can read it
X = torch.tensor(df_balanced['Encoded_Sequence'].tolist(), dtype=torch.long)

# y MUST be torch.float32 for the Binary Cross Entropy (BCE) Loss function
y = torch.tensor(df_balanced['Label'].tolist(), dtype=torch.float32)

# Create the PyTorch DataLoader (This feeds the data to the Neural Network in batches of 64)
BATCH_SIZE = 64
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"--- PYTORCH TENSORS READY ---")
print(f"X Tensor Shape (Input):  {X.shape}  -> (Rows, Sequence_Length)")
print(f"y Tensor Shape (Labels): {y.shape}  -> (Rows, 1)")
print(f"Data type of X: {X.dtype}")
print(f"sequence preview:\n{X[1]}")

--- PYTORCH TENSORS READY ---
X Tensor Shape (Input):  torch.Size([59634, 100])  -> (Rows, Sequence_Length)
y Tensor Shape (Labels): torch.Size([59634])  -> (Rows, 1)
Data type of X: torch.int64
sequence preview:
tensor([ 4,  4,  2,  2, 17, 17, 16, 16, 16, 10,  6,  6, 16, 16,  1,  6, 16,  9,
         9,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0])


In [25]:
# ==========================================
# STEP 11: Define the LSTM Architecture
# ==========================================
class TraceAnomalyLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers=1, dropout=0.2):
        super(TraceAnomalyLSTM, self).__init__()
        
        # 1. Embedding Layer
        # padding_idx=0 tells PyTorch to ignore all those 0s in your sequences!
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 2. LSTM Layer
        # batch_first=True tells PyTorch our data is formatted as (Batch, Sequence, Features)
        self.lstm = nn.LSTM(embedding_dim, 
                            hidden_dim, 
                            num_layers=num_layers, 
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        
        # 3. Fully Connected (Linear) Layer
        self.fc = nn.Linear(hidden_dim, output_dim)
        
        # 4. Sigmoid Activation (Outputs a probability between 0 and 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # x shape: (batch_size, sequence_length) -> e.g., (64, 100)
        
        # Step A: Convert integers to vectors
        embedded = self.embedding(x)
        
        # Step B: Pass through LSTM
        # 'hidden' contains the final memory state of the entire sequence
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # Step C: Grab the memory from the very last LSTM layer
        final_hidden_state = hidden[-1]
        
        # Step D: Pass through the linear layer and squash to a probability
        out = self.fc(final_hidden_state)
        return self.sigmoid(out).squeeze()

print("--- AI ARCHITECTURE DEFINED ---")

--- AI ARCHITECTURE DEFINED ---


In [26]:
# ==========================================
# STEP 12: Initialize and Test the Model
# ==========================================
# Find the maximum integer in our dataset to set the Vocabulary Size dynamically
# We add +1 because 0-indexing means the max integer is one less than the size
VOCAB_SIZE = int(X.max()) + 1  
EMBEDDING_DIM = 32  # Size of the vector for each token
HIDDEN_DIM = 64     # Size of the LSTM's memory
OUTPUT_DIM = 1      # We just want 1 number (the probability)

# Initialize the model
model = TraceAnomalyLSTM(vocab_size=VOCAB_SIZE, 
                         embedding_dim=EMBEDDING_DIM, 
                         hidden_dim=HIDDEN_DIM, 
                         output_dim=OUTPUT_DIM)

# Grab exactly ONE batch of data from our dataloader
sample_X, sample_y = next(iter(dataloader))

# Push it through the untrained network!
dummy_predictions = model(sample_X)

print(f"--- DUMMY PASS SUCCESSFUL ---")
print(f"Input Batch Shape: {sample_X.shape}")
print(f"Output Predictions Shape: {dummy_predictions.shape}")
print(f"First 5 Untrained Predictions:\n{dummy_predictions[:5].detach().numpy()}")

--- DUMMY PASS SUCCESSFUL ---
Input Batch Shape: torch.Size([64, 100])
Output Predictions Shape: torch.Size([64])
First 5 Untrained Predictions:
[0.5288983 0.5288983 0.5288983 0.5288983 0.5288983]


In [28]:

# ==========================================
# STEP 13: Define Loss and Optimizer
# ==========================================
# Binary Cross Entropy (BCE) - The mathematical standard for 0/1 classification
criterion = nn.BCELoss() 

# Adam Optimizer - The algorithm that updates the weights
# lr=0.001 is the "Learning Rate" (how big of a step it takes when fixing mistakes)
optimizer = optim.Adam(model.parameters(), lr=0.001)
# ==========================================
# STEP 14: The Training Loop
# ==========================================
EPOCHS = 5  # Number of times to loop through the entire dataset

print("--- STARTING TRAINING ---")
for epoch in range(EPOCHS):
    model.train() # Put model in training mode
    
    epoch_loss = 0
    correct_predictions = 0
    total_predictions = 0
    
    # Loop through the data in batches of 64
    for batch_X, batch_y in dataloader:
        
        # 1. Zero out the gradients from the previous step
        optimizer.zero_grad()
        
        # 2. Forward pass: Make predictions (The dummy pass we just did)
        predictions = model(batch_X)
        
        # 3. Calculate Loss: How wrong were we?
        loss = criterion(predictions, batch_y)
        
        # 4. Backward pass: Calculate the math to fix the mistakes
        loss.backward()
        
        # 5. Optimizer Step: Actually update the LSTM weights
        optimizer.step()
        
        # --- Tracking Stats for the Console ---
        epoch_loss += loss.item()
        
        # If probability > 0.5, guess 1 (Anomaly), else 0 (Normal)
        rounded_preds = torch.round(predictions)
        correct_predictions += (rounded_preds == batch_y).sum().item()
        total_predictions += len(batch_y)
        
    # Print the results at the end of each epoch
    epoch_accuracy = (correct_predictions / total_predictions) * 100
    avg_loss = epoch_loss / len(dataloader)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Accuracy: {epoch_accuracy:.2f}%")
    
print("--- TRAINING COMPLETE ---")

--- STARTING TRAINING ---
Epoch 1/5 | Loss: 0.6917 | Accuracy: 51.50%
Epoch 2/5 | Loss: 0.6904 | Accuracy: 50.52%
Epoch 3/5 | Loss: 0.3104 | Accuracy: 89.01%
Epoch 4/5 | Loss: 0.2332 | Accuracy: 92.47%
Epoch 5/5 | Loss: 0.2180 | Accuracy: 93.28%
--- TRAINING COMPLETE ---


In [32]:
# ==========================================
# STEP 15: Export the Trained AI (Corrected)
# ==========================================
# Create a folder to store our saved models if it doesn't exist
os.makedirs('../../src/models/saved', exist_ok=True)

MODEL_PATH = '../../src/models/saved/trace_anomaly_lstm.pth'

# Save ONLY the Model Weights (The Math)
torch.save(model.state_dict(), MODEL_PATH)

print("--- EXPORT COMPLETE ---")
print(f"Model saved to: {MODEL_PATH}")
print("(Note: Vocab export skipped because HDFS_v3 data was pre-tokenized!)")

--- EXPORT COMPLETE ---
Model saved to: ../../src/models/saved/trace_anomaly_lstm.pth
(Note: Vocab export skipped because HDFS_v3 data was pre-tokenized!)
